In [1]:
%cd /content/imdb-sentiment
%pip install -e .

/content/imdb-sentiment
Obtaining file:///content/imdb-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imdb-sentiment (pyproject.toml) ... done
  Created wheel for imdb-sentiment: filename=imdb_sentiment-0.1.0-0.editable-py3-none-any.whl size=5195 sha256=45c36d3a8776266f6bf99b82d50d226a1773c2c1a47068adcd6969e4cd6a5e71
  Stored in directory: /tmp/pip-ephem-wheel-cache-jhrug657/wheels/3e/08/09/1f8f2dc306b20c0f6d4e5a4ddb080c355c7ce988d70820356f
Successfully built imdb-sentiment
  Attempting uninstall: imdb-sentiment
    Found existing installation: imdb-sentiment 0.1.0
    Uninstalling imdb-sentiment-0.1.0:
      Successfully uninstalled imdb-sentiment-0.1.0


In [2]:
import sys
sys.path.append("/content/imdb-sentiment/src")

In [3]:
import imdb_sentiment
print("Project import: OK")

Project import: OK


In [4]:
from pathlib import Path

import torch
import pandas as pd
import numpy as np

from datasets import load_dataset

sys.path.append("/content/imdb-sentiment/src")


In [5]:
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [6]:
from statistics import mean
from collections import Counter
from pathlib import Path
import re

from imdb_sentiment.data.dataset import load_imdb_dataset
from imdb_sentiment.data.lstm import build_lstm_vocabulary, tokenize_lstm_text
from imdb_sentiment.settings import load_config, LSTMModelConfig

In [7]:
dataset = load_imdb_dataset()
train_split = dataset["train"]

sample_size = 5000
texts = list(train_split.select(range(sample_size))["text"])

token_lengths = [len(tokenize_lstm_text(text)) for text in texts]
sorted_lengths = sorted(token_lengths)

def percentile(values, p):
    idx = int(len(values) * p)
    idx = min(idx, len(values) - 1)
    return values[idx]

max_length = 256
num_truncated = sum(length > max_length for length in token_lengths)

print("=== LSTM TOKEN LENGTH STATS ===")
print("sample size:", len(token_lengths))
print("min:", min(token_lengths))
print("mean:", round(mean(token_lengths), 2))
print("p50:", percentile(sorted_lengths, 0.50))
print("p90:", percentile(sorted_lengths, 0.90))
print("p95:", percentile(sorted_lengths, 0.95))
print("p99:", percentile(sorted_lengths, 0.99))
print("max:", max(token_lengths))

print("\n=== TRUNCATION CHECK ===")
print("max_length:", max_length)
print("num texts longer than max_length:", num_truncated)
print("share truncated:", round(num_truncated / len(token_lengths), 4))

bucket_counts = Counter()
for length in token_lengths:
    if length <= 64:
        bucket_counts["0-64"] += 1
    elif length <= 128:
        bucket_counts["65-128"] += 1
    elif length <= 256:
        bucket_counts["129-256"] += 1
    elif length <= 512:
        bucket_counts["257-512"] += 1
    else:
        bucket_counts["513+"] += 1

print("\n=== LENGTH BUCKETS ===")
for bucket, count in bucket_counts.items():
    print(bucket, ":", count)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


=== LSTM TOKEN LENGTH STATS ===
sample size: 5000
min: 10
mean: 228.09
p50: 172
p90: 437
p95: 571
p99: 905
max: 1491

=== TRUNCATION CHECK ===
max_length: 256
num texts longer than max_length: 1390
share truncated: 0.278

=== LENGTH BUCKETS ===
257-512 : 1054
129-256 : 2322
65-128 : 1069
513+ : 336
0-64 : 219


In [8]:
candidate_lengths = [128, 256, 384, 512, 640]

print("=== TRUNCATION BY CANDIDATE max_length ===")
for candidate in candidate_lengths:
    num_truncated = sum(length > candidate for length in token_lengths)
    share_truncated = num_truncated / len(token_lengths)
    print(
        f"max_length={candidate:<3} | "
        f"num_truncated={num_truncated:<4} | "
        f"share_truncated={share_truncated:.4f}"
    )

=== TRUNCATION BY CANDIDATE max_length ===
max_length=128 | num_truncated=3712 | share_truncated=0.7424
max_length=256 | num_truncated=1390 | share_truncated=0.2780
max_length=384 | num_truncated=646  | share_truncated=0.1292
max_length=512 | num_truncated=336  | share_truncated=0.0672
max_length=640 | num_truncated=174  | share_truncated=0.0348


In [9]:
config_path = Path("/content/imdb-sentiment/configs/experiments/lstm_baseline_v1.yaml")

text = config_path.read_text(encoding="utf-8")
text = text.replace("  max_length: 256", "  max_length: 512")

config_path.write_text(text, encoding="utf-8")

print(config_path.read_text(encoding="utf-8"))

experiment:
  family: lstm
  name: baseline_v1

seed: 42

paths:
  model_output: artifacts/experiments/lstm/baseline_v1/model.pt
  val_metrics_output: artifacts/experiments/lstm/baseline_v1/val_metrics.json
  test_metrics_output: artifacts/experiments/lstm/baseline_v1/test_metrics.json
# Standardized LSTM bundle in this directory:
# required for predict/evaluate input -> model.pt, vocab.json, training_config.json
# produced by training -> model.pt, vocab.json, training_config.json, val_metrics.json
# produced by evaluation -> test_metrics.json

model:
  type: lstm
  vocab_size: 30000
  max_length: 512
  embedding_dim: 128
  hidden_dim: 128
  batch_size: 32
  epochs: 5
  dropout: 0.3
  lr: 0.001



In [10]:
def set_yaml_vocab_size(config_path: Path, vocab_size: int) -> None:
    text = config_path.read_text(encoding="utf-8")
    text = re.sub(
        r"(^\s*vocab_size:\s*)\d+",
        rf"\g<1>{vocab_size}",
        text,
        flags=re.MULTILINE,
    )
    config_path.write_text(text, encoding="utf-8")

def measure_lstm_vocab_coverage(config_path: Path, dataset, sample_limit: int = 5000) -> dict:
    config = load_config(config_path)
    assert isinstance(config.model, LSTMModelConfig)

    train_val_split = dataset["train"].train_test_split(
        test_size=0.2,
        seed=config.seed,
    )

    train_texts = list(train_val_split["train"]["text"])
    val_texts = list(train_val_split["test"]["text"])

    vocab = build_lstm_vocabulary(
        texts=train_texts,
        max_size=config.model.vocab_size,
    )

    unk_id = vocab.unk_id

    total_tokens = 0
    unk_tokens = 0
    texts_checked = 0
    texts_with_unk = 0

    for text in val_texts[:sample_limit]:
        tokens = tokenize_lstm_text(text)
        if not tokens:
            continue

        token_ids = [vocab.lookup_token_id(token) for token in tokens]
        unk_count = sum(token_id == unk_id for token_id in token_ids)

        total_tokens += len(token_ids)
        unk_tokens += unk_count
        texts_checked += 1

        if unk_count > 0:
            texts_with_unk += 1

    result = {
        "configured_vocab_size": config.model.vocab_size,
        "actual_vocab_size": len(vocab),
        "unk_share": round(unk_tokens / total_tokens, 4) if total_tokens else 0.0,
        "texts_with_unk_share": round(texts_with_unk / texts_checked, 4) if texts_checked else 0.0,
    }
    return result

In [11]:
original_config_text = config_path.read_text(encoding="utf-8")

set_yaml_vocab_size(config_path, 20000)
coverage_20000 = measure_lstm_vocab_coverage(config_path, dataset)

In [12]:
set_yaml_vocab_size(config_path, 30000)
coverage_30000 = measure_lstm_vocab_coverage(config_path, dataset)

In [13]:
print("coverage_20000:", coverage_20000)
print("coverage_30000:", coverage_30000)

coverage_20000: {'configured_vocab_size': 20000, 'actual_vocab_size': 20000, 'unk_share': 0.0848, 'texts_with_unk_share': 0.9934}
coverage_30000: {'configured_vocab_size': 30000, 'actual_vocab_size': 30000, 'unk_share': 0.0678, 'texts_with_unk_share': 0.9876}


In [14]:
def audit_lstm_vocabulary_cleanliness(config_path, dataset, train_sample_limit=5000):
    config = load_config(config_path)
    assert isinstance(config.model, LSTMModelConfig)

    train_val_split = dataset["train"].train_test_split(
        test_size=0.2,
        seed=config.seed,
    )
    train_texts = list(train_val_split["train"]["text"])

    vocab = build_lstm_vocabulary(
        texts=train_texts,
        max_size=config.model.vocab_size,
    )

    all_tokens = list(vocab.token_to_id.keys())
    special_tokens = {"<pad>", "<unk>"}
    regular_tokens = [tok for tok in all_tokens if tok not in special_tokens]

    patterns = {
        "html_like": re.compile(r"<[^>]+>|&[a-z]+;"),
        "url_like": re.compile(r"(http|www\.|\.com|\.net|\.org)"),
        "only_punct": re.compile(r"^[^\w\s]+$"),
        "contains_digit": re.compile(r"\d"),
        "very_long": re.compile(r"^.{25,}$"),
        "repeated_symbol": re.compile(r"(.)\1{3,}"),
    }

    flagged = {name: [] for name in patterns}

    for token in regular_tokens:
        for name, pattern in patterns.items():
            if pattern.search(token):
                flagged[name].append(token)

    token_counter = Counter()
    for text in train_texts[:train_sample_limit]:
        token_counter.update(tokenize_lstm_text(text))

    result = {
        "configured_vocab_size": config.model.vocab_size,
        "actual_vocab_size": len(vocab),
        "regular_token_count": len(regular_tokens),
        "flagged_counts": {name: len(items) for name, items in flagged.items()},
        "top_40_vocab_tokens": regular_tokens[:40],
        "flagged_token_samples": {name: items[:30] for name, items in flagged.items()},
        "most_common_train_tokens": token_counter.most_common(40),
    }

    print("=== VOCAB CLEANLINESS SUMMARY ===")
    print("configured_vocab_size:", result["configured_vocab_size"])
    print("actual_vocab_size:", result["actual_vocab_size"])
    print("regular_token_count:", result["regular_token_count"])

    print("\n=== FLAGGED COUNTS ===")
    for name, count in result["flagged_counts"].items():
        print(f"{name}: {count}")

    print("\n=== TOP 40 TOKENS IN VOCAB ===")
    print(result["top_40_vocab_tokens"])

    print("\n=== FLAGGED TOKEN SAMPLES ===")
    for name, items in result["flagged_token_samples"].items():
        print(f"\n{name}:")
        print(items)

    print("\n=== MOST COMMON TOKENS IN TRAIN SAMPLE ===")
    print(result["most_common_train_tokens"])

    return result

In [15]:
vocab_audit = audit_lstm_vocabulary_cleanliness(config_path, dataset)

=== VOCAB CLEANLINESS SUMMARY ===
configured_vocab_size: 30000
actual_vocab_size: 30000
regular_token_count: 29998

=== FLAGGED COUNTS ===
html_like: 0
url_like: 2
only_punct: 63
contains_digit: 574
very_long: 0
repeated_symbol: 16

=== TOP 40 TOKENS IN VOCAB ===
['the', 'a', 'and', 'of', 'to', 'is', 'in', 'i', 'this', 'that', 'it', 'was', 'as', 'for', 'with', 'but', 'on', 'movie', 'his', 'not', 'are', 'film', 'you', 'have', 'he', 'be', 'at', 'one', 'by', 'an', 'they', 'from', 'all', 'who', 'like', 'so', 'just', 'her', 'or', 'has']

=== FLAGGED TOKEN SAMPLES ===

html_like:
[]

url_like:
['imdb.com', 'amazon.com']

only_punct:
['-', '&', '--', '\x96', '.', ',', '!', '...', '"', '(', '?', ':', '****', '/', '*', ')', ':)', '***', '**', '–', '..', '=', '---', '*****', '+', '~', ').', '!!', '....', '\x97']

contains_digit:
['2', '10', '3', '1', '4', '5', '20', '10.', '30', '15', '8', '7', "80's", '9', '90', '6', '40', '100', '2.', "70's", '1.', '12', '10/10', '50', '3.', '11', '8/10', '80s